In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import yaml
from IPython.display import Markdown, display
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
from scipy.stats import linregress
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, roc_auc_score, accuracy_score, confusion_matrix
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# load yaml
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)
    DATA_PATH = config.get("data_path")
    if DATA_PATH is None:
        print("ERROR: No data path provided")
    USE_DRIVE = bool(config.get("use_drive", False))
# load from drive if requested
if USE_DRIVE:
    from google.colab import drive

    drive.mount('/content/drive')

In [23]:
def make_window_df(df, features, window):
    """
    Given a DataFrame with one row per team-game, compute:
      - own-team rolling averages of `metrics` over `window` games
      - opponent-team rolling averages of the same `metrics` over `window` games

    Parameters
    ----------
    df : pd.DataFrame
      Must contain columns:
        - 'gameDate' (datetime convertible)
        - 'teamName', 'opponentTeamName'
    features : list of str
      e.g. ['teamScore','threePointersPercentage', …]
    window : int
      The look-back window size (e.g. 15)

    Returns
    -------
    pd.DataFrame
      Copy of `df` with new columns:
        - '{feature}_avg_{window}'
        - 'opp_{feature}_avg_{window}'
    """
    # 1) Prep & season
    df = df.copy()
    df['gameDate'] = pd.to_datetime(df['gameDate'])
    df = df.sort_values(['teamName','gameDate'])
    df['season'] = np.where(
        df['gameDate'].dt.month >= 10,
        df['gameDate'].dt.year,
        df['gameDate'].dt.year - 1
    )

    # 2) Own-team rolling averages
    for feat in features:
        col = f'{feat}_avg_{window}'
        df[col] = (
            df
            .groupby(['teamName','season'])[feat]
            .transform(lambda x: x.shift().rolling(window).mean())
        )

    # 3) Opponent-team rolling averages
    #    First build a tiny lookup DF of opponents' past form:
    df_opp = (
    df[['gameDate','season','opponentTeamName'] + features]
    .rename(columns={'opponentTeamName':'teamName'})
    .sort_values(['teamName','gameDate'])      # ← THIS IS CRUCIAL
    )
    for feat in features:
        df_opp[f'opp_{feat}_avg_{window}'] = (
            df_opp
            .groupby(['teamName','season'])[feat]
            .transform(lambda x: x.shift().rolling(window).mean())
        )

    # 4) Merge those back onto the main DF
    keep = ['gameDate','season','teamName'] + [f'opp_{feat}_avg_{window}' for feat in features]
    df = df.merge(
        df_opp[keep],
        left_on=['gameDate','season','opponentTeamName'],
        right_on=['gameDate','season','teamName'],
        how='left',
        suffixes=('','_drop')
    ).drop(columns=['teamName_drop'])

    return df


In [24]:
important_features = pd.read_csv(os.path.join(DATA_PATH, f"important_features.csv"))
important_features.columns

Index(['gameId', 'gameDate', 'teamCity', 'teamName', 'opponentTeamCity',
       'opponentTeamName', 'home', 'win', 'teamScore', 'opponentScore',
       'threePointersPercentage', 'freeThrowsPercentage',
       'assistsPerPossession', 'blocksPerPossession', 'stealsPerPossession',
       'threePointersAttemptedPerPossession',
       'freeThrowsAttemptedPerPossession', 'reboundsDefensivePerPossession',
       'reboundsOffensivePerPossession', 'foulsPersonalPerPossession',
       'turnoversPerPossession', 'effectiveFieldGoalPercentage',
       'trueShootingPercentage'],
      dtype='object')

In [25]:
important_features = pd.read_csv(os.path.join(DATA_PATH, "important_features.csv"))

features = [
    'teamScore',
    'threePointersPercentage',
    'freeThrowsPercentage',
    'assistsPerPossession',
    'blocksPerPossession',
    'stealsPerPossession',
    'threePointersAttemptedPerPossession',
    'freeThrowsAttemptedPerPossession',
    'reboundsDefensivePerPossession',
    'reboundsOffensivePerPossession',
    'foulsPersonalPerPossession',
    'turnoversPerPossession',
    'effectiveFieldGoalPercentage',
    'trueShootingPercentage'
]

df_15_avg = make_window_df(important_features, features, window=15)

In [26]:
df_15_avg.head()

,gameId,gameDate,teamCity,teamName,opponentTeamCity,opponentTeamName,home,win,teamScore,opponentScore,...,opp_blocksPerPossession_avg_15,opp_stealsPerPossession_avg_15,opp_threePointersAttemptedPerPossession_avg_15,opp_freeThrowsAttemptedPerPossession_avg_15,opp_reboundsDefensivePerPossession_avg_15,opp_reboundsOffensivePerPossession_avg_15,opp_foulsPersonalPerPossession_avg_15,opp_turnoversPerPossession_avg_15,opp_effectiveFieldGoalPercentage_avg_15,opp_trueShootingPercentage_avg_15
0,29600003,1996-11-01,Philadelphia,76ers,Milwaukee,Bucks,1,0,103,111.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,29600019,1996-11-02,Philadelphia,76ers,Chicago,Bulls,0,0,86,115.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,29600032,1996-11-05,Philadelphia,76ers,Detroit,Pistons,1,0,81,83.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,29600053,1996-11-08,Philadelphia,76ers,Boston,Celtics,0,1,115,105.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,29600063,1996-11-09,Philadelphia,76ers,Phoenix,Suns,1,1,112,95.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
df_15_avg.to_csv(os.path.join(DATA_PATH, "rolling_avg/TEST_df_15_avg.csv"))

In [28]:
windows = [1, 3, 5, 10, 15, 20, 30]

for window in windows:
    df_window = make_window_df(important_features, features, window=window)
    df_window.to_csv(os.path.join(DATA_PATH, f"rolling_avg/team_stats_{window}_avg.csv"))

In [29]:
df_window.columns

Index(['gameId', 'gameDate', 'teamCity', 'teamName', 'opponentTeamCity',
       'opponentTeamName', 'home', 'win', 'teamScore', 'opponentScore',
       'threePointersPercentage', 'freeThrowsPercentage',
       'assistsPerPossession', 'blocksPerPossession', 'stealsPerPossession',
       'threePointersAttemptedPerPossession',
       'freeThrowsAttemptedPerPossession', 'reboundsDefensivePerPossession',
       'reboundsOffensivePerPossession', 'foulsPersonalPerPossession',
       'turnoversPerPossession', 'effectiveFieldGoalPercentage',
       'trueShootingPercentage', 'season', 'teamScore_avg_30',
       'threePointersPercentage_avg_30', 'freeThrowsPercentage_avg_30',
       'assistsPerPossession_avg_30', 'blocksPerPossession_avg_30',
       'stealsPerPossession_avg_30',
       'threePointersAttemptedPerPossession_avg_30',
       'freeThrowsAttemptedPerPossession_avg_30',
       'reboundsDefensivePerPossession_avg_30',
       'reboundsOffensivePerPossession_avg_30',
       'foulsPersonalP